In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
ingestion_url = "s3://knowledge-intelligence-github-issues/raw/github-issues/*.json"
df = spark.read.option("multiLine", "true").json(ingestion_url)
df.createOrReplaceTempView("github_issues")
df = spark.sql("""SELECT 
               id, -- pk,
               number, --github issue number
               title,
               body, -- important for rag
               state, -- open/close
               state_reason, -- reason for open/close
               created_at, -- how many issues created each day
               closed_at, -- how many issues closed each day
               labels, -- bug/prod/critical
               comments, -- number of comments
               comments_url, -- use API call to get comments
               events_url,
               labels_url
               FROM github_issues""")
display(df)
print(df.columns)

id,number,title,body,state,state_reason,created_at,closed_at,labels,comments,comments_url,events_url,labels_url
5557895567,3,Test Issue 3,I am test issue 3,open,null,2026-09-23T18:43:22Z,null,"List(List(null, null, f143ab, true, Barrier affecting people with disabilities, 12276633894, accessibility, LA_kwDOUijy7M8AAAAC276RJg, https://api.github.com/repos/utkarshadl1702/Enterprise-Incident-Knowledge-Intelligence-Platform/labels/accessibility), List(null, null, 008672, true, Extra attention is needed, 12276633923, help wanted, LA_kwDOUijy7M8AAAAC276RQw, https://api.github.com/repos/utkarshadl1702/Enterprise-Incident-Knowledge-Intelligence-Platform/labels/help%20wanted), List(null, null, e4e669, true, This doesn't seem right, 12276633933, invalid, LA_kwDOUijy7M8AAAAC276RTQ, https://api.github.com/repos/utkarshadl1702/Enterprise-Incident-Knowledge-Intelligence-Platform/labels/invalid))",0,https://api.github.com/repos/utkarshadl1702/Enterprise-Incident-Knowledge-Intelligence-Platform/issues/3/comments,https://api.github.com/repos/utkarshadl1702/Enterprise-Incident-Knowledge-Intelligence-Platform/issues/3/events,https://api.github.com/repos/utkarshadl1702/Enterprise-Incident-Knowledge-Intelligence-Platform/issues/3/labels{/name}
5518550716,1,Test Issue,Test issue to check working of github API and data ingestion. Data ingestion working fine.,open,null,2026-09-20T14:55:20Z,null,List(),0,https://api.github.com/repos/utkarshadl1702/Enterprise-Incident-Knowledge-Intelligence-Platform/issues/1/comments,https://api.github.com/repos/utkarshadl1702/Enterprise-Incident-Knowledge-Intelligence-Platform/issues/1/events,https://api.github.com/repos/utkarshadl1702/Enterprise-Incident-Knowledge-Intelligence-Platform/issues/1/labels{/name}
5518550716,1,Test Issue,Test issue to check working of github API and data ingestion. Data ingestion working fine.,open,null,2026-09-20T14:55:20Z,null,List(),0,https://api.github.com/repos/utkarshadl1702/Enterprise-Incident-Knowledge-Intelligence-Platform/issues/1/comments,https://api.github.com/repos/utkarshadl1702/Enterprise-Incident-Knowledge-Intelligence-Platform/issues/1/events,https://api.github.com/repos/utkarshadl1702/Enterprise-Incident-Knowledge-Intelligence-Platform/issues/1/labels{/name}


['id', 'number', 'title', 'body', 'state', 'state_reason', 'created_at', 'closed_at', 'labels', 'comments', 'comments_url', 'events_url', 'labels_url']


<bound method DataFrame.describe of DataFrame[active_lock_reason: string, assignee: string, assignees: array<string>, author_association: string, body: string, closed_at: string, closed_by: string, comments: bigint, comments_url: string, created_at: string, events_url: string, html_url: string, id: bigint, issue_dependencies_summary: struct<blocked_by:bigint,blocking:bigint,total_blocked_by:bigint,total_blocking:bigint>, labels: array<string>, labels_url: string, locked: boolean, milestone: string, node_id: string, number: bigint, performed_via_github_app: string, pinned_comment: string, reactions: struct<+1:bigint,-1:bigint,confused:bigint,eyes:bigint,heart:bigint,hooray:bigint,laugh:bigint,rocket:bigint,total_count:bigint,url:string>, repository_url: string, state: string, state_reason: string, sub_issues_summary: struct<completed:bigint,percent_completed:bigint,total:bigint>, timeline_url: string, title: string, updated_at: string, url: string, user: struct<avatar_url:string,events_

In [0]:
dbutils.widgets.text("catalog","ekip","Catalog")
dbutils.widgets.text("data_source","br_gitdata","Data Source")
dbutils.widgets.text("layer","bronze","Layer")

catalog=dbutils.widgets.get("catalog")
data_source=dbutils.widgets.get("data_source")
layer = dbutils.widgets.get("layer")
print(catalog,data_source,layer)

ekip br_gitdata bronze


Dump Relevant columns into Bronze layer

In [0]:
bronze_table = f"{catalog}.{layer}.{data_source}"

In [0]:
df.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("append") \
 .saveAsTable(bronze_table)

In [0]:
df_bronze = spark.sql(f"""SELECT * FROM {catalog}.bronze.{data_source}""")
display(df_bronze)

id,number,title,body,state,state_reason,created_at,closed_at,labels,comments,comments_url,events_url,labels_url
5518550716,1,Test Issue,Test issue to check working of github API and data ingestion. Data ingestion working fine.,open,null,2026-09-20T14:55:20Z,null,List(),0,https://api.github.com/repos/utkarshadl1702/Enterprise-Incident-Knowledge-Intelligence-Platform/issues/1/comments,https://api.github.com/repos/utkarshadl1702/Enterprise-Incident-Knowledge-Intelligence-Platform/issues/1/events,https://api.github.com/repos/utkarshadl1702/Enterprise-Incident-Knowledge-Intelligence-Platform/issues/1/labels{/name}
5518550716,1,Test Issue,Test issue to check working of github API and data ingestion. Data ingestion working fine.,open,null,2026-09-20T14:55:20Z,null,List(),0,https://api.github.com/repos/utkarshadl1702/Enterprise-Incident-Knowledge-Intelligence-Platform/issues/1/comments,https://api.github.com/repos/utkarshadl1702/Enterprise-Incident-Knowledge-Intelligence-Platform/issues/1/events,https://api.github.com/repos/utkarshadl1702/Enterprise-Incident-Knowledge-Intelligence-Platform/issues/1/labels{/name}
5557895567,3,Test Issue 3,I am test issue 3,open,null,2026-09-23T18:43:22Z,null,"List(List(null, null, f143ab, true, Barrier affecting people with disabilities, 12276633894, accessibility, LA_kwDOUijy7M8AAAAC276RJg, https://api.github.com/repos/utkarshadl1702/Enterprise-Incident-Knowledge-Intelligence-Platform/labels/accessibility), List(null, null, 008672, true, Extra attention is needed, 12276633923, help wanted, LA_kwDOUijy7M8AAAAC276RQw, https://api.github.com/repos/utkarshadl1702/Enterprise-Incident-Knowledge-Intelligence-Platform/labels/help%20wanted), List(null, null, e4e669, true, This doesn't seem right, 12276633933, invalid, LA_kwDOUijy7M8AAAAC276RTQ, https://api.github.com/repos/utkarshadl1702/Enterprise-Incident-Knowledge-Intelligence-Platform/labels/invalid))",0,https://api.github.com/repos/utkarshadl1702/Enterprise-Incident-Knowledge-Intelligence-Platform/issues/3/comments,https://api.github.com/repos/utkarshadl1702/Enterprise-Incident-Knowledge-Intelligence-Platform/issues/3/events,https://api.github.com/repos/utkarshadl1702/Enterprise-Incident-Knowledge-Intelligence-Platform/issues/3/labels{/name}
